# Variant parsing and sequence reconstruction

This notebook demonstrates the core parsing and sequence reconstruction utilities provided by `dms_parser`.

We will cover:

1. Parsing internal protein variant notation such as `A23V`
2. Parsing multi-mutants such as `A23V;G45D`
3. Parsing MaveDB-style `hgvs_pro` strings
4. Reconstructing mutated sequences from a wild-type protein sequence
5. Inspecting common edge cases and unsupported variants

In [1]:
import pandas as pd

from dms_parser import (
    apply_mutations,
    count_mutations,
    explode_mutations,
    extract_mutation_tokens,
    extract_positions,
    hgvs_pro_is_indel,
    hgvs_to_sequence,
    parse_hgvs_pro,
    parse_mavedb_hgvs_series,
    parse_variant,
    parse_variant_series,
    parse_variant_token,
    variant_to_sequence,
)

## 1. Define a wild-type protein sequence

We will use a short wild-type sequence for the reconstruction examples.

In [2]:
wt_sequence = "MKTAYIAKQRQISFVKSHFSRQDILDLWQ"
wt_sequence

'MKTAYIAKQRQISFVKSHFSRQDILDLWQ'

## 2. Parse a single internal variant token

Internal variant tokens follow the common one-letter amino-acid notation:

- `M1A`
- `K2R`
- `T3Y`

In [3]:
parse_variant_token("M1A")

{'token': 'M1A', 'wt_aa': 'M', 'position': 1, 'mut_aa': 'A'}

## 3. Parse a full internal variant

The parser supports:

- wild type
- single mutants
- multi-mutants

In [4]:
examples = ["WT", "M1A", "K2R", "M1A;K2R"]

for variant in examples:
    print(f"Variant: {variant}")
    print(parse_variant(variant))
    print("-" * 80)

Variant: WT
{'variant': 'WT', 'is_wildtype': True, 'n_mutations': 0, 'mutations': [], 'position': None, 'wt_aa': None, 'mut_aa': None}
--------------------------------------------------------------------------------
Variant: M1A
{'variant': 'M1A', 'is_wildtype': False, 'n_mutations': 1, 'mutations': [{'token': 'M1A', 'wt_aa': 'M', 'position': 1, 'mut_aa': 'A'}], 'position': 1, 'wt_aa': 'M', 'mut_aa': 'A'}
--------------------------------------------------------------------------------
Variant: K2R
{'variant': 'K2R', 'is_wildtype': False, 'n_mutations': 1, 'mutations': [{'token': 'K2R', 'wt_aa': 'K', 'position': 2, 'mut_aa': 'R'}], 'position': 2, 'wt_aa': 'K', 'mut_aa': 'R'}
--------------------------------------------------------------------------------
Variant: M1A;K2R
{'variant': 'M1A;K2R', 'is_wildtype': False, 'n_mutations': 2, 'mutations': [{'token': 'M1A', 'wt_aa': 'M', 'position': 1, 'mut_aa': 'A'}, {'token': 'K2R', 'wt_aa': 'K', 'position': 2, 'mut_aa': 'R'}], 'position': None,

## 4. Count mutations and extract mutation information

In [5]:
variant = "M1A;K2R"

print("Mutation count:", count_mutations(variant))
print("Positions:", extract_positions(variant))
print("Tokens:", extract_mutation_tokens(variant))

Mutation count: 2
Positions: [1, 2]
Tokens: ['M1A', 'K2R']


## 5. Parse a full pandas Series of internal variants

In [6]:
series = pd.Series(["WT", "M1A", "K2R", "M1A;K2R", "T3Y"])
parsed_df = parse_variant_series(series, prefix="parsed_")
parsed_df

,parsed_variant,parsed_is_wildtype,parsed_n_mutations,parsed_mutations,parsed_position,parsed_wt_aa,parsed_mut_aa
0,WT,True,0,[],NaN,None,None
1,M1A,False,1,"[{'token': 'M1A', 'wt_aa': 'M', 'position': 1,...",1.0,M,A
2,K2R,False,1,"[{'token': 'K2R', 'wt_aa': 'K', 'position': 2,...",2.0,K,R
3,M1A;K2R,False,2,"[{'token': 'M1A', 'wt_aa': 'M', 'position': 1,...",NaN,None,None
4,T3Y,False,1,"[{'token': 'T3Y', 'wt_aa': 'T', 'position': 3,...",3.0,T,Y


## 6. Explode multi-mutants

For some analyses, it is useful to convert a list of mutations into one row per mutation.

In [7]:
exploded_df = explode_mutations(
    parsed_df.rename(columns={"parsed_mutations": "mutations"}),
    mutations_col="mutations",
)

exploded_df

,parsed_variant,parsed_is_wildtype,parsed_n_mutations,mutations,parsed_position,parsed_wt_aa,parsed_mut_aa,token,wt_aa,position,mut_aa
0,WT,True,0,NaN,NaN,None,None,None,None,NaN,None
1,M1A,False,1,"{'token': 'M1A', 'wt_aa': 'M', 'position': 1, ...",1.0,M,A,M1A,M,1.0,A
2,K2R,False,1,"{'token': 'K2R', 'wt_aa': 'K', 'position': 2, ...",2.0,K,R,K2R,K,2.0,R
3,M1A;K2R,False,2,"{'token': 'M1A', 'wt_aa': 'M', 'position': 1, ...",NaN,None,None,M1A,M,1.0,A
3,M1A;K2R,False,2,"{'token': 'K2R', 'wt_aa': 'K', 'position': 2, ...",NaN,None,None,K2R,K,2.0,R
4,T3Y,False,1,"{'token': 'T3Y', 'wt_aa': 'T', 'position': 3, ...",3.0,T,Y,T3Y,T,3.0,Y


## 7. Parse HGVS protein variants

MaveDB often stores variants using `hgvs_pro` notation, for example:

- `p.Met1Ala`
- `p.[Met1Ala;Lys2Arg]`
- `p.Met1=`

In [8]:
hgvs_examples = [
    "p.Met1Ala",
    "p.[Met1Ala;Lys2Arg]",
    "p.Met1=",
]

for hgvs in hgvs_examples:
    print(f"HGVS: {hgvs}")
    print(parse_hgvs_pro(hgvs))
    print("-" * 80)

HGVS: p.Met1Ala
[('M', 1, 'A')]
--------------------------------------------------------------------------------
HGVS: p.[Met1Ala;Lys2Arg]
[('M', 1, 'A'), ('K', 2, 'R')]
--------------------------------------------------------------------------------
HGVS: p.Met1=
[('M', 1, 'M')]
--------------------------------------------------------------------------------


## 8. Detect unsupported HGVS patterns

Protein-level indels, duplications, frameshifts, and extensions are not supported in the substitutions-only workflow.

In [9]:
unsupported_examples = [
    "p.Gly10del",
    "p.Arg15dup",
    "p.Leu20fs",
    "p.Met1Ala",
]

for hgvs in unsupported_examples:
    print(hgvs, "->", hgvs_pro_is_indel(hgvs))

p.Gly10del -> True
p.Arg15dup -> True
p.Leu20fs -> True
p.Met1Ala -> False


## 9. Parse a Series of `hgvs_pro` values

This is especially useful when exploring MaveDB score tables before full dataset construction.

In [10]:
hgvs_series = pd.Series(
    [
        "p.Met1Ala",
        "p.[Met1Ala;Lys2Arg]",
        "p.Gly10del",
        "p.invalid",
    ]
)

parsed_hgvs_df = parse_mavedb_hgvs_series(hgvs_series, strict=False)
parsed_hgvs_df

,hgvs_pro,status,error,mutations
0,p.Met1Ala,OK,,"[(M, 1, A)]"
1,p.[Met1Ala;Lys2Arg],OK,,"[(M, 1, A), (K, 2, R)]"
2,p.Gly10del,Unsupported,Protein-level indel/frameshift not supported i...,None
3,p.invalid,Error,Unsupported mutation token: invalid in p.invalid,None


## 10. Reconstruct a mutated sequence from internal notation

The function `variant_to_sequence(...)` applies the parsed substitutions to the wild-type sequence.

In [11]:
mutated_sequence, normalized_variant = variant_to_sequence(wt_sequence, "M1A")
print("Normalized variant:", normalized_variant)
print("Mutated sequence:  ", mutated_sequence)

Normalized variant: M1A
Mutated sequence:   AKTAYIAKQRQISFVKSHFSRQDILDLWQ


## 11. Reconstruct a mutated sequence from a multi-mutant

In [12]:
mutated_sequence, normalized_variant = variant_to_sequence(wt_sequence, "M1A;K2R")
print("Normalized variant:", normalized_variant)
print("Mutated sequence:  ", mutated_sequence)

Normalized variant: M1A;K2R
Mutated sequence:   ARTAYIAKQRQISFVKSHFSRQDILDLWQ


## 12. Reconstruct a mutated sequence from HGVS protein notation

In [13]:
mutated_sequence, normalized_variant = hgvs_to_sequence(wt_sequence, "p.[Met1Ala;Lys2Arg]")
print("Normalized variant:", normalized_variant)
print("Mutated sequence:  ", mutated_sequence)

Normalized variant: M1A;K2R
Mutated sequence:   ARTAYIAKQRQISFVKSHFSRQDILDLWQ


## 13. Apply mutations directly

At the lowest level, the library can apply a list of substitutions directly to the WT sequence.

In [14]:
mutations = [("M", 1, "A"), ("K", 2, "R")]
mutated_sequence, normalized_variant = apply_mutations(wt_sequence, mutations)

print("Normalized variant:", normalized_variant)
print("Mutated sequence:  ", mutated_sequence)

Normalized variant: M1A;K2R
Mutated sequence:   ARTAYIAKQRQISFVKSHFSRQDILDLWQ


## 14. Compare WT and reconstructed variants

In [15]:
comparison_df = pd.DataFrame(
    {
        "variant": ["WT", "M1A", "M1A;K2R", "T3Y"],
        "mutated_sequence": [
            wt_sequence,
            variant_to_sequence(wt_sequence, "M1A")[0],
            variant_to_sequence(wt_sequence, "M1A;K2R")[0],
            variant_to_sequence(wt_sequence, "T3Y")[0],
        ],
    }
)

comparison_df

,variant,mutated_sequence
0,WT,MKTAYIAKQRQISFVKSHFSRQDILDLWQ
1,M1A,AKTAYIAKQRQISFVKSHFSRQDILDLWQ
2,M1A;K2R,ARTAYIAKQRQISFVKSHFSRQDILDLWQ
3,T3Y,MKYAYIAKQRQISFVKSHFSRQDILDLWQ


## 15. Inspect an invalid internal variant

This example shows how parsing behaves for malformed variant strings.

In [16]:
try:
    parse_variant("BADVARIANT")
except Exception as exc:
    print(type(exc).__name__)
    print(str(exc))

InvalidVariantError
Invalid substitution token: 'BADVARIANT'


## 16. Inspect an unsupported HGVS variant

In [17]:
try:
    parse_hgvs_pro("p.Gly10del")
except Exception as exc:
    print(type(exc).__name__)
    print(str(exc))

UnsupportedVariantError
Unsupported hgvs_pro (protein-level indel/frameshift/etc.) for substitutions-only analysis.


## 17. Inspect a WT mismatch during reconstruction

Sequence reconstruction also validates that the expected WT residue matches the provided WT sequence.

In [18]:
try:
    apply_mutations(wt_sequence, [("A", 1, "V")])
except Exception as exc:
    print(type(exc).__name__)
    print(str(exc))

MutationApplicationError
WT mismatch at pos 1: mutation expects A, WT has M.


## 18. Summary

In this notebook we demonstrated the main low-level utilities of `dms_parser`:

- parsing internal one-letter protein variants
- parsing MaveDB `hgvs_pro` substitutions
- identifying unsupported HGVS patterns
- reconstructing mutated protein sequences from a WT sequence
- validating and applying substitutions

These functions form the core logic used by the higher-level dataset builders.